# 4.5. Enhanced Labeling Review

Interactive labeling tool that works from the cluster inventory produced by `4. Auto-Labeling Pipeline.ipynb`.

**Three modes:**
- **Confirm** — cluster has auto-label (frac ≥ 0.60): one click to confirm or override
- **Review** — cluster has weak auto-label (0.40–0.60): show prior label, full selector
- **Unknown** — no prior label: full manual labeling

**Extra features over 3.5:**
- Live class distribution bar chart with per-class target counts
- Jump to most under-labeled class
- `+ New class` button to register custom label codes at runtime
- Split workflow (same as 3.5) — drag range sliders to define sub-regions
- Partial car hint for thin elongated clusters

**Output:** updates `CLUSTERS_DIR/inventory.csv` with `final_label` and `label_source_final` columns.
Sub-region splits are saved as extra rows in the inventory.

## Config

In [36]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('.').resolve()))

from config import CLUSTERS_DIR

INVENTORY_CSV = CLUSTERS_DIR / "inventory.csv"
CUSTOM_LABELS_JSON = CLUSTERS_DIR / "custom_labels.json"

# Per-class labeling targets (adjust as you go)
LABEL_TARGETS = {
    30: 400,   # Tree
    40: 400,   # Car
    49:  50,   # Partial car
    60: 200,   # Street light
    80: 150,   # City bench
    81: 150,   # Rubbish bin
    83: 150,   # Large container
    44: 200,   # Single bike
    46: 200,   # Multiple bikes
    47: 100,   # Bike on pole
    45: 150,   # Scooter
    65: 200,   # Slender pole
    99:  50,   # Noise
}

AUTO_LABEL_THRESH = 0.60
REVIEW_THRESH     = 0.40
PC_PADDING        = 0.15

## Load inventory and custom labels

In [37]:
import numpy as np
import pandas as pd
import json
from datetime import datetime

import laspy

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import io

import ipywidgets as widgets
from IPython.display import display
from utils.labels import Labels

from config import CLUSTERS_DIR, LABELED_DIR

# Load inventory
inv = pd.read_csv(INVENTORY_CSV)
if 'final_label' not in inv.columns:
    inv['final_label'] = None
if 'label_source_final' not in inv.columns:
    inv['label_source_final'] = None
print(f"Loaded {len(inv)} clusters from {INVENTORY_CSV}")

# Load (or create) custom labels registry
if CUSTOM_LABELS_JSON.exists():
    with open(CUSTOM_LABELS_JSON) as f:
        custom_labels = {int(k): v for k, v in json.load(f).items()}
else:
    custom_labels = {}

def all_label_names():
    """Merge Labels.STR_DICT with custom_labels."""
    merged = dict(Labels.STR_DICT)
    merged.update(custom_labels)
    return merged

def save_inventory():
    inv.to_csv(INVENTORY_CSV, index=False)

def save_custom_labels():
    with open(CUSTOM_LABELS_JSON, 'w') as f:
        json.dump(custom_labels, f)

# Sort: confirm first, then review, then unknown
def sort_key(row):
    if row['final_label'] is not None and not pd.isna(row['final_label']):
        return 3  # already labeled → last
    frac = float(row['label_frac']) if not pd.isna(row['label_frac']) else 0.0
    if frac >= AUTO_LABEL_THRESH:
        return 0  # confirm mode
    if frac >= REVIEW_THRESH:
        return 1  # review mode
    return 2      # unknown

inv = inv.iloc[inv.apply(sort_key, axis=1).argsort(kind='stable')].reset_index(drop=True)

# Count already labeled
labeled_mask = inv['final_label'].notna()
print(f"  Already labeled: {labeled_mask.sum()}  |  Remaining: {(~labeled_mask).sum()}")

Loaded 63 clusters from data/output/clusters/inventory.csv
  Already labeled: 0  |  Remaining: 63


## Core label set and colors

In [32]:
# Initial label set (code, name, hex color) — extended dynamically
BASE_LABEL_SET = [
    (40,  'Car',           '#d62728'),
    (49,  'Partial car',   '#ff9896'),
    (30,  'Tree',          '#2ca02c'),
    (60,  'Street light',  '#ff7f0e'),
    (80,  'City bench',    '#9467bd'),
    (81,  'Rubbish bin',   '#8c564b'),
    (83,  'Container',     '#e377c2'),
    (44,  'Single bike',   '#aec7e8'),
    (46,  'Multi-bike',    '#1f77b4'),
    (47,  'Bike on pole',  '#17becf'),
    (45,  'Scooter',       '#bcbd22'),
    (65,  'Slender pole',  '#7f7f7f'),
    (99,  'Noise',         '#444444'),
]
LABEL_COLOR_MAP = {name: color for _, name, color in BASE_LABEL_SET}
LABEL_CODE_MAP  = {name: code  for code, name, _ in BASE_LABEL_SET}
CODE_COLOR_MAP  = {code: color for code, _, color in BASE_LABEL_SET}

BGT_COLORS = {
     1: '#f5f500',  9: '#99cc99', 10: '#ff8800', 30: '#2ca02c',
    40: '#d62728', 60: '#ff7f0e', 65: '#7f7f7f', 70: '#aaaaaa',
    79: '#cccccc', 83: '#8c564b', 99: '#333333',
}

## Render helpers

In [33]:
def _load_cluster(row):
    """Load cluster data from NPZ. Returns (xyz_c, rgb, height_ag, centroid_xy)."""
    data = np.load(row['npz_path'], allow_pickle=False)
    xyz_c = data['xyz_centered']  # (N, 3), XY centered
    rgb   = data['rgb_norm']      # (N, 3)
    h_ag  = data['height_ag']     # (N,)
    cxy   = data['centroid_xy']   # (2,)
    return xyz_c, rgb, h_ag, cxy

def _fig_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='#1a1a1a')
    plt.close(fig); buf.seek(0)
    return buf.read()

def _sa(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.tick_params(labelsize=6, colors='grey')

# ── Tile context ───────────────────────────────────────────────────────────────
_tile_cache = {}  # tilecode -> (x, y, tile_labels) or None

# Road=1 yellow, Ground=9 green, unknown=very dark, building=medium grey
TILE_LABEL_COLORS = {0: '#1e1e1e', 1: '#e8d44d', 9: '#6aaa6a', 10: '#666666'}

def _load_tile_context(tilecode):
    if tilecode in _tile_cache:
        return _tile_cache[tilecode]
    laz_path = LABELED_DIR / f"road_ground_labeled_{tilecode}.laz"
    if not laz_path.exists():
        _tile_cache[tilecode] = None
        return None
    laz  = laspy.read(str(laz_path))
    step = max(1, len(laz.x) // 80_000)  # target ~80k points for speed
    x      = np.array(laz.x[::step],         dtype=np.float32)
    y      = np.array(laz.y[::step],         dtype=np.float32)
    labels = np.array(laz['label'][::step],   dtype=np.uint8)
    _tile_cache[tilecode] = (x, y, labels)
    return x, y, labels

def render_tile_context(current_row):
    tilecode  = current_row['tilecode']
    result    = _load_tile_context(tilecode)
    tile_inv  = inv[inv['tilecode'] == tilecode]
    cur_cidx  = current_row['cluster_idx']

    fig, ax = plt.subplots(figsize=(5, 5))
    fig.patch.set_facecolor('#1a1a1a')
    _sa(ax)

    if result is None:
        ax.text(0.5, 0.5, f'Tile LAZ not found:\n{tilecode}',
                color='white', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Tile context', color='#888', fontsize=8)
        plt.tight_layout(pad=0.3)
        return _fig_png(fig)

    x, y, tile_labels = result

    # Ground layer: unknown (very faint), road, ground
    for lbl in np.unique(tile_labels):
        mask   = tile_labels == lbl
        color  = TILE_LABEL_COLORS.get(int(lbl), '#444444')
        alpha  = 0.12 if lbl == 0 else 0.65
        size   = 0.2  if lbl == 0 else 0.7
        ax.scatter(x[mask], y[mask], c=color, s=size, linewidths=0,
                   alpha=alpha, rasterized=True, zorder=1)

    # Unlabeled clusters (excluding current)
    unlabeled = tile_inv[tile_inv['final_label'].isna() &
                         (tile_inv['cluster_idx'] != cur_cidx)]
    if len(unlabeled):
        ax.scatter(unlabeled['centroid_x'].astype(float),
                   unlabeled['centroid_y'].astype(float),
                   c='#555555', s=25, linewidths=0, zorder=3, marker='o')

    # Labeled clusters
    lnames = all_label_names()
    labeled = tile_inv[tile_inv['final_label'].notna()]
    for _, row in labeled.iterrows():
        code  = int(row['final_label'])
        color = CODE_COLOR_MAP.get(code, '#bbbbbb')
        cx, cy = float(row['centroid_x']), float(row['centroid_y'])
        ax.scatter(cx, cy, c=color, s=55, zorder=4,
                   edgecolors='white', linewidths=0.5, marker='o')
        name = lnames.get(code, str(code))[:9]
        ax.annotate(name, (cx, cy), textcoords='offset points', xytext=(4, 3),
                    fontsize=5, color=color, zorder=5)

    # Current cluster: white ring + red centre dot
    cx = float(current_row['centroid_x'])
    cy = float(current_row['centroid_y'])
    ax.scatter(cx, cy, c='none', s=220, zorder=9,
               edgecolors='red', linewidths=2, marker='o')
    ax.scatter(cx, cy, c='red', s=18, zorder=10, linewidths=0, marker='o')

    ax.set_aspect('equal')
    ax.set_title(f'Tile: {tilecode}', color='#aaa', fontsize=8)
    ax.set_xlabel('X  (RD New)', color='grey', fontsize=6)
    ax.set_ylabel('Y  (RD New)', color='grey', fontsize=6)

    legend_patches = [
        mpatches.Patch(color='#e8d44d', label='Road'),
        mpatches.Patch(color='#6aaa6a', label='Ground'),
        mpatches.Patch(color='#555555', label='Unlabeled cluster'),
    ]
    ax.legend(handles=legend_patches, loc='lower left', fontsize=5,
              facecolor='#2a2a2a', edgecolor='#555', labelcolor='white',
              markerscale=0.7)

    plt.tight_layout(pad=0.3)
    return _fig_png(fig)

def render_label_view(row, mode):
    xyz_c, rgb, h_ag, cxy = _load_cluster(row)
    bx0, bx1 = float(xyz_c[:, 0].min()), float(xyz_c[:, 0].max())
    by0, by1 = float(xyz_c[:, 1].min()), float(xyz_c[:, 1].max())
    pad = PC_PADDING

    fig, (ax_t, ax_s) = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor('#1a1a1a')
    _sa(ax_t); _sa(ax_s)

    ax_t.scatter(xyz_c[:, 0], xyz_c[:, 1], c=rgb, s=8, linewidths=0, zorder=2)
    ax_t.set_aspect('equal')
    ax_t.set_xlim(bx0 - pad, bx1 + pad)
    ax_t.set_ylim(by0 - pad, by1 + pad)

    # Existing BGT label title
    auto_lbl  = int(row['label']) if not pd.isna(row['label']) else 0
    label_frac = float(row['label_frac']) if not pd.isna(row['label_frac']) else 0.0
    if auto_lbl > 0:
        lname = all_label_names().get(auto_lbl, str(auto_lbl))
        col   = BGT_COLORS.get(auto_lbl, '#ffffff')
        ax_t.set_title(f'Auto-label: {lname}  ({label_frac:.0%})', color=col, fontsize=8)
    else:
        ax_t.set_title('Top view', color='#888', fontsize=8)

    # Partial car hint
    dx = bx1 - bx0; dy = by1 - by0
    aspect = max(dx, dy) / max(min(dx, dy), 0.01)
    max_h = float(xyz_c[:, 2].max() - xyz_c[:, 2].min())
    if aspect > 5 and max_h < 0.8:
        ax_t.set_xlabel('Scan sliver? Consider Partial Car', color='#ff9896', fontsize=7)

    ax_s.scatter(xyz_c[:, 0], xyz_c[:, 2], c=rgb, s=8, linewidths=0)
    ax_s.set_xlim(bx0 - pad, bx1 + pad)
    ax_s.set_title('Side profile', color='#888', fontsize=8)
    ax_s.set_xlabel('X (centered)', color='grey', fontsize=7)
    ax_s.set_ylabel('Z (absolute)', color='grey', fontsize=7)
    _sa(ax_s)

    mode_colors = {'confirm': '#2ca02c', 'review': '#ff7f0e', 'unknown': '#aaaaaa'}
    fig.suptitle(f"Mode: {mode.upper()}  |  tile: {row['tilecode']}  |  cluster #{row['cluster_idx']}",
                 color=mode_colors.get(mode, 'white'), fontsize=9)
    plt.tight_layout(pad=0.5)
    return _fig_png(fig)

def render_class_distribution():
    labeled = inv[inv['final_label'].notna()]
    if len(labeled) == 0:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.set_facecolor('#1a1a1a'); fig.patch.set_facecolor('#1a1a1a')
        ax.text(0.5, 0.5, 'No labels yet', color='white', ha='center', va='center', transform=ax.transAxes)
        return _fig_png(fig)

    lnames = all_label_names()
    counts = labeled['final_label'].value_counts()
    codes  = counts.index.tolist()
    vals   = counts.values.tolist()
    targets = [LABEL_TARGETS.get(int(c), 0) for c in codes]
    colors  = [CODE_COLOR_MAP.get(int(c), '#bbbbbb') for c in codes]
    names   = [lnames.get(int(c), str(c)) for c in codes]

    fig, ax = plt.subplots(figsize=(5, max(3, len(codes) * 0.4)))
    fig.patch.set_facecolor('#1a1a1a')
    ax.set_facecolor('#1a1a1a')

    y = np.arange(len(codes))
    ax.barh(y, vals, color=colors, alpha=0.85)
    for i, (v, t) in enumerate(zip(vals, targets)):
        if t > 0:
            ax.axvline(t, ymin=(i) / len(codes), ymax=(i + 1) / len(codes),
                       color='white', linewidth=0.8, alpha=0.5, linestyle='--')
        ax.text(v + 1, i, str(v), color='white', va='center', fontsize=7)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=7, color='white')
    ax.set_xlabel('count', color='grey', fontsize=7)
    ax.tick_params(colors='grey', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.set_title('Label distribution (dashed = target)', color='white', fontsize=8)
    plt.tight_layout(pad=0.3)
    return _fig_png(fig)

def render_split_preview(row, regions):
    xyz_c, rgb, _, _ = _load_cluster(row)
    bx0, bx1 = float(xyz_c[:, 0].min()), float(xyz_c[:, 0].max())
    by0, by1 = float(xyz_c[:, 1].min()), float(xyz_c[:, 1].max())
    pad = PC_PADDING * 3
    clrs = ['#ffff00', '#00ff88', '#ff8800', '#ff44ff']

    fig, ax = plt.subplots(figsize=(7, 7))
    fig.patch.set_facecolor('#1a1a1a'); _sa(ax)
    ax.scatter(xyz_c[:, 0], xyz_c[:, 1], c=rgb, s=8, linewidths=0, zorder=2)
    for i, r in enumerate(regions):
        xmn, xmx, ymn, ymx = r
        col = clrs[i % len(clrs)]
        ax.add_patch(plt.Rectangle((xmn, ymn), xmx - xmn, ymx - ymn,
                                   facecolor=col, alpha=0.2, edgecolor=col,
                                   linewidth=1.5, zorder=4))
        ax.text((xmn+xmx)/2, (ymn+ymx)/2, str(i+1),
                color=col, fontsize=14, fontweight='bold', ha='center', va='center', zorder=5)
    ax.set_aspect('equal')
    ax.set_xlim(bx0 - pad, bx1 + pad); ax.set_ylim(by0 - pad, by1 + pad)
    ax.set_title(f'Split preview ({len(regions)} regions)', color='white', fontsize=10)
    plt.tight_layout()
    return _fig_png(fig)

def render_subregion_view(row, si, regions):
    xyz_c, rgb, _, _ = _load_cluster(row)
    bx0, bx1 = float(xyz_c[:, 0].min()), float(xyz_c[:, 0].max())
    by0, by1 = float(xyz_c[:, 1].min()), float(xyz_c[:, 1].max())
    pad = PC_PADDING * 2
    xmn, xmx, ymn, ymx = regions[si]
    in_r = ((xyz_c[:, 0] >= xmn) & (xyz_c[:, 0] <= xmx) &
            (xyz_c[:, 1] >= ymn) & (xyz_c[:, 1] <= ymx))

    fig, (ax_t, ax_s) = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor('#1a1a1a'); _sa(ax_t); _sa(ax_s)
    ax_t.scatter(xyz_c[~in_r, 0], xyz_c[~in_r, 1], c=rgb[~in_r], s=5, linewidths=0, alpha=0.12, zorder=2)
    ax_t.scatter(xyz_c[in_r,  0], xyz_c[in_r,  1], c=rgb[in_r],  s=10, linewidths=0, zorder=3)
    ax_t.add_patch(plt.Rectangle((xmn, ymn), xmx-xmn, ymx-ymn,
                                 facecolor='none', edgecolor='yellow', linewidth=1.5, zorder=4))
    ax_t.set_aspect('equal')
    ax_t.set_xlim(bx0 - pad, bx1 + pad); ax_t.set_ylim(by0 - pad, by1 + pad)
    ax_t.set_title(f'Sub-region {si+1}/{len(regions)}', color='yellow', fontsize=9)
    ax_s.scatter(xyz_c[~in_r, 0], xyz_c[~in_r, 2], c=rgb[~in_r], s=5, linewidths=0, alpha=0.12)
    ax_s.scatter(xyz_c[in_r,  0], xyz_c[in_r,  2], c=rgb[in_r],  s=10, linewidths=0)
    ax_s.set_xlim(bx0 - pad, bx1 + pad)
    ax_s.set_title('Side profile', color='#888', fontsize=8)
    plt.tight_layout(pad=0.5)
    return _fig_png(fig)

## Label widgets and state

In [34]:
def _get_mode(row):
    if row['final_label'] is not None and not pd.isna(row['final_label']):
        return 'done'
    frac = float(row['label_frac']) if not pd.isna(row['label_frac']) else 0.0
    if frac >= AUTO_LABEL_THRESH:
        return 'confirm'
    if frac >= REVIEW_THRESH:
        return 'review'
    return 'unknown'

# State
state = {
    'idx': 0,
    'widget_mode': 'label',  # 'label' | 'split_define' | 'split_label'
    'split_regions': [],
    'split_subidx': 0,
    'split_pending': {},     # row_idx -> [{xmin,xmax,ymin,ymax,label_id,label_name}]
}
# Find first unlabeled
for i, row in inv.iterrows():
    if _get_mode(row) != 'done':
        state['idx'] = i; break

# ── Build label buttons ───────────────────────────────────────────────────────
def _make_lbl_btns(label_set):
    btns = []
    for code, name, color in label_set:
        b = widgets.Button(
            description=name,
            layout=widgets.Layout(width='118px', height='30px'),
            style={'button_color': color},
        )
        b._lid = code; b._lname = name
        btns.append(b)
    return btns

lbl_btns = _make_lbl_btns(BASE_LABEL_SET)
half     = (len(lbl_btns) + 1) // 2
lbl_row1 = widgets.HBox(lbl_btns[:half])
lbl_row2 = widgets.HBox(lbl_btns[half:])
lbl_panel = widgets.VBox([lbl_row1, lbl_row2])

# ── Navigation & action buttons ───────────────────────────────────────────────
prev_btn       = widgets.Button(description='← Prev',        button_style='info',    layout=widgets.Layout(width='80px'))
skip_btn       = widgets.Button(description='Skip',          button_style='warning', layout=widgets.Layout(width='65px'))
next_btn       = widgets.Button(description='Next →',        button_style='info',    layout=widgets.Layout(width='80px'))
confirm_btn    = widgets.Button(description='✓ Confirm',     button_style='success', layout=widgets.Layout(width='90px'))
split_btn      = widgets.Button(description='✂ Split',       layout=widgets.Layout(width='75px'))
jump_box       = widgets.BoundedIntText(value=0, min=0, max=len(inv)-1, description='#:', layout=widgets.Layout(width='110px'))
jump_btn       = widgets.Button(description='Go',            layout=widgets.Layout(width='45px'))
jump_class_btn = widgets.Button(description='→ Underrepresented', button_style='info', layout=widgets.Layout(width='175px'))
noise_btn      = widgets.Button(description='Noise',         button_style='danger',  layout=widgets.Layout(width='65px'))

# ── Progress ──────────────────────────────────────────────────────────────────
progress_confirm = widgets.IntProgress(min=0, max=max(1, len(inv[inv.apply(lambda r: _get_mode(r) == 'confirm', axis=1)])),
                                       bar_style='success', description='Confirm', layout=widgets.Layout(width='180px'))
progress_review  = widgets.IntProgress(min=0, max=max(1, len(inv[inv.apply(lambda r: _get_mode(r) in ('confirm','review'), axis=1)])),
                                       bar_style='warning', description='Review',  layout=widgets.Layout(width='180px'))
progress_total   = widgets.IntProgress(min=0, max=len(inv), bar_style='info',     description='Total',   layout=widgets.Layout(width='180px'))

# ── Info + plots ──────────────────────────────────────────────────────────────
info      = widgets.HTML()
plot_img  = widgets.Image(format='png', width=900)
tile_img  = widgets.Image(format='png', width=500)  # tile context view
dist_img  = widgets.Image(format='png', width=400)

# ── Custom label panel ────────────────────────────────────────────────────────
new_name_box = widgets.Text(placeholder='New class name…', layout=widgets.Layout(width='160px'))
new_code_box = widgets.BoundedIntText(value=200, min=200, max=999, description='Code:', layout=widgets.Layout(width='110px'))
new_btn      = widgets.Button(description='+ Add class', button_style='', layout=widgets.Layout(width='100px'))
new_status   = widgets.HTML()
custom_panel = widgets.HBox([new_name_box, new_code_box, new_btn, new_status])

# ── Split panel (mirrors 3.5) ─────────────────────────────────────────────────
split_regions_vbox = widgets.VBox([])
split_regions_vbox._sx = []; split_regions_vbox._sy = []

add_region_btn    = widgets.Button(description='+ Region',      layout=widgets.Layout(width='90px'))
split_preview_btn = widgets.Button(description='Preview',       button_style='info',    layout=widgets.Layout(width='80px'))
split_apply_btn   = widgets.Button(description='Label regions', button_style='success', layout=widgets.Layout(width='120px'))
split_cancel_btn  = widgets.Button(description='Cancel',        button_style='warning', layout=widgets.Layout(width='75px'))

split_panel = widgets.VBox([
    widgets.HTML('<b style="color:#ffd700">Drag sliders to define rectangular sub-regions</b>'),
    split_regions_vbox,
    widgets.HBox([add_region_btn, split_preview_btn, split_apply_btn, split_cancel_btn]),
], layout=widgets.Layout(border='1px solid #555', padding='8px'))
split_panel.layout.display = 'none'

subregion_info    = widgets.HTML()
cancel_split_btn  = widgets.Button(description='✕ Cancel split', button_style='warning', layout=widgets.Layout(width='115px'))
subregion_nav     = widgets.HBox([cancel_split_btn])
subregion_nav.layout.display = 'none'

main_ui = widgets.VBox([
    info,
    widgets.HBox([plot_img, tile_img, dist_img]),
    lbl_panel,
    widgets.HBox([prev_btn, skip_btn, next_btn, confirm_btn, noise_btn, split_btn,
                  progress_total, jump_box, jump_btn, jump_class_btn]),
    widgets.HBox([progress_confirm, progress_review]),
    custom_panel,
    split_panel,
    subregion_info,
    subregion_nav,
])

print("Widgets built.")

Widgets built.


TypeError: Invalid value 'auto_confirmed' for dtype 'float64'

TypeError: Invalid value 'manual' for dtype 'float64'

TypeError: Invalid value 'auto_confirmed' for dtype 'float64'

TypeError: Invalid value 'auto_confirmed' for dtype 'float64'

...

TypeError: Invalid value 'auto_confirmed' for dtype 'float64'

TypeError: Invalid value 'manual' for dtype 'float64'

TypeError: Invalid value 'manual' for dtype 'float64'

TypeError: Invalid value 'manual' for dtype 'float64'

TypeError: Invalid value 'manual' for dtype 'float64'

## Event handlers and refresh

In [35]:
def _advance():
    for i in range(state['idx'] + 1, len(inv)):
        if _get_mode(inv.iloc[i]) != 'done':
            state['idx'] = i; return

def _apply_label(row_idx, label_code, label_name, source):
    inv.at[row_idx, 'final_label']        = label_code
    inv.at[row_idx, 'label_source_final'] = source
    inv.at[row_idx, 'timestamp']          = datetime.now().isoformat(timespec='seconds')
    save_inventory()

def _do_label(code, name):
    idx = state['idx']
    if state['widget_mode'] == 'label':
        _apply_label(idx, code, name, 'manual')
        _advance(); refresh()
    elif state['widget_mode'] == 'split_label':
        si = state['split_subidx']
        state['split_pending'][idx][si]['label_id']   = code
        state['split_pending'][idx][si]['label_name'] = name
        # Add sub-region as a separate inventory row
        parent = inv.iloc[idx]
        reg    = state['split_pending'][idx][si]
        new_row = parent.to_dict()
        new_row.update({
            'cluster_idx':  f"{parent['cluster_idx']}_sub{si}",
            'final_label':  code,
            'label_source_final': 'manual_split',
            'timestamp': datetime.now().isoformat(timespec='seconds'),
        })
        inv.loc[len(inv)] = new_row
        if si < len(state['split_pending'][idx]) - 1:
            state['split_subidx'] += 1
        else:
            _apply_label(idx, -1, 'split', 'split')
            state['widget_mode'] = 'label'
            _advance()
        save_inventory()
        refresh()

def on_label(btn): _do_label(btn._lid, btn._lname)

def on_confirm(b):
    idx  = state['idx']
    row  = inv.iloc[idx]
    code = int(row['label']) if not pd.isna(row['label']) else 0
    if code > 0:
        name = all_label_names().get(code, str(code))
        _apply_label(idx, code, name, 'auto_confirmed')
        _advance(); refresh()

def on_noise(b): _do_label(99, 'Noise')

def on_next(b):
    state['widget_mode'] = 'label'
    if state['idx'] < len(inv) - 1: state['idx'] += 1
    refresh()

def on_prev(b):
    state['widget_mode'] = 'label'
    if state['idx'] > 0: state['idx'] -= 1
    refresh()

def on_jump(b):
    state['widget_mode'] = 'label'
    state['idx'] = int(jump_box.value)
    refresh()

def on_jump_class(b):
    """Jump to first unlabeled cluster of the most under-represented class."""
    labeled = inv[inv['final_label'].notna()]
    counts  = labeled['final_label'].value_counts()
    worst   = None; worst_ratio = 2.0
    for code, target in LABEL_TARGETS.items():
        if target == 0: continue
        ratio = counts.get(code, 0) / target
        if ratio < worst_ratio:
            worst_ratio = ratio; worst = code
    if worst is None: return
    # Find first unlabeled cluster likely to be this class (auto-label matches)
    for i, row in inv.iterrows():
        if _get_mode(row) == 'done': continue
        if int(row['label']) if not pd.isna(row['label']) else 0 == worst:
            state['idx'] = i; state['widget_mode'] = 'label'; refresh(); return
    # Otherwise just jump to first unlabeled of any mode
    _advance(); refresh()

def on_new_class(b):
    name = new_name_box.value.strip()
    code = int(new_code_box.value)
    if not name:
        new_status.value = '<span style="color:red">Enter a class name.</span>'; return
    custom_labels[code] = name
    save_custom_labels()
    # Add button to lbl_panel
    colors = ['#bbbbbb', '#f4c542', '#aaffaa', '#ffaaaa']
    color  = colors[len(custom_labels) % len(colors)]
    CODE_COLOR_MAP[code] = color
    b2 = widgets.Button(description=name, layout=widgets.Layout(width='118px', height='30px'),
                        style={'button_color': color})
    b2._lid = code; b2._lname = name
    b2.on_click(on_label)
    lbl_row2.children = list(lbl_row2.children) + [b2]
    new_name_box.value = ''
    new_status.value = f'<span style="color:#2ca02c">Added: {name} (code {code})</span>'

# ── Split workflow ─────────────────────────────────────────────────────────────
def _gbounds():
    row  = inv.iloc[state['idx']]
    data = np.load(row['npz_path'], allow_pickle=False)
    pts  = data['xyz_centered']
    xmn, xmx = float(pts[:, 0].min()), float(pts[:, 0].max())
    ymn, ymx = float(pts[:, 1].min()), float(pts[:, 1].max())
    xp = max(0.05, (xmx - xmn) * 0.2); yp = max(0.05, (ymx - ymn) * 0.2)
    return xmn - xp, xmx + xp, ymn - yp, ymx + yp

def _read_regions():
    return [[sx.value[0], sx.value[1], sy.value[0], sy.value[1]]
            for sx, sy in zip(split_regions_vbox._sx, split_regions_vbox._sy)]

def _rebuild_sliders(regions_init):
    gxmn, gxmx, gymn, gymx = _gbounds()
    rows, new_sx, new_sy = [], [], []
    for i, r in enumerate(regions_init):
        sx = widgets.FloatRangeSlider(value=[r[0], r[1]], min=gxmn, max=gxmx, step=0.02,
                                      description=f'R{i+1} X:', readout_format='.2f',
                                      continuous_update=False, layout=widgets.Layout(width='430px'))
        sy = widgets.FloatRangeSlider(value=[r[2], r[3]], min=gymn, max=gymx, step=0.02,
                                      description=f'     Y:', readout_format='.2f',
                                      continuous_update=False, layout=widgets.Layout(width='430px'))
        rm = widgets.Button(description='✕', button_style='danger',
                            layout=widgets.Layout(width='33px', height='26px'))
        rm._ri = i; rm.on_click(_on_rm_region)
        rows.append(widgets.HBox([widgets.VBox([sx, sy]), rm]))
        new_sx.append(sx); new_sy.append(sy)
    split_regions_vbox._sx = new_sx
    split_regions_vbox._sy = new_sy
    split_regions_vbox.children = rows

def on_split(b):
    row = inv.iloc[state['idx']]
    data = np.load(row['npz_path'], allow_pickle=False)
    pts = data['xyz_centered']
    r = [float(pts[:, 0].min()), float(pts[:, 0].max()),
         float(pts[:, 1].min()), float(pts[:, 1].max())]
    _rebuild_sliders([r])
    state['widget_mode'] = 'split_define'; refresh()

def _on_rm_region(b):
    regs = _read_regions(); regs.pop(b._ri)
    if not regs: state['widget_mode'] = 'label'; refresh(); return
    _rebuild_sliders(regs)
    state['split_regions'] = _read_regions(); refresh()

def on_add_region(b):
    if len(split_regions_vbox._sx) >= 4: return
    regs = _read_regions()
    row  = inv.iloc[state['idx']]
    data = np.load(row['npz_path'], allow_pickle=False)
    pts  = data['xyz_centered']
    xmid = float((pts[:, 0].min() + pts[:, 0].max()) / 2)
    regs.append([xmid, float(pts[:, 0].max()),
                 float(pts[:, 1].min()), float(pts[:, 1].max())])
    _rebuild_sliders(regs)

def on_split_preview(b):
    state['split_regions'] = _read_regions()
    plot_img.value = render_split_preview(inv.iloc[state['idx']], state['split_regions'])

def on_split_apply(b):
    regs = _read_regions()
    if not regs: return
    idx = state['idx']
    state['split_pending'][idx] = [
        {'xmin': r[0], 'xmax': r[1], 'ymin': r[2], 'ymax': r[3],
         'label_id': None, 'label_name': None} for r in regs
    ]
    state['split_regions'] = regs
    state['split_subidx']  = 0
    state['widget_mode']   = 'split_label'
    refresh()

def on_split_cancel(b):    state['widget_mode'] = 'label'; refresh()
def on_cancel_split_lb(b):
    state['split_pending'].pop(state['idx'], None)
    state['widget_mode'] = 'label'; refresh()

# ── Refresh ────────────────────────────────────────────────────────────────────
def refresh():
    idx  = state['idx']
    row  = inv.iloc[idx]
    mode = _get_mode(row) if state['widget_mode'] == 'label' else state['widget_mode']

    done = int(inv['final_label'].notna().sum())
    n_confirm = int((inv.apply(lambda r: _get_mode(r) == 'confirm', axis=1)).sum())
    n_rev     = int((inv.apply(lambda r: _get_mode(r) in ('confirm', 'review'), axis=1)).sum())

    auto_lbl = int(row['label']) if not pd.isna(row['label']) else 0
    lname    = all_label_names().get(auto_lbl, '?') if auto_lbl else ''

    info.value = (
        f"<b>#{idx}  ({idx+1}/{len(inv)})</b>  |  "
        f"mode: <b style='color:#ffd700'>{mode}</b>  |  "
        f"auto: {lname} ({float(row['label_frac']) if not pd.isna(row['label_frac']) else 0:.0%})  |  "
        f"labeled: {done}/{len(inv)}"
    )
    jump_box.value        = idx
    progress_total.value  = done
    progress_confirm.value = n_confirm
    progress_review.value  = n_rev

    if state['widget_mode'] == 'label':
        plot_img.value = render_label_view(row, mode)
        tile_img.value = render_tile_context(row)
        split_panel.layout.display   = 'none'
        subregion_info.value         = ''
        subregion_nav.layout.display = 'none'
    elif state['widget_mode'] == 'split_define':
        state['split_regions'] = _read_regions()
        plot_img.value = render_split_preview(row, state['split_regions'])
        tile_img.value = render_tile_context(row)
        split_panel.layout.display   = ''
        subregion_info.value         = ''
        subregion_nav.layout.display = 'none'
    elif state['widget_mode'] == 'split_label':
        si = state['split_subidx']
        plot_img.value = render_subregion_view(row, si, state['split_regions'])
        tile_img.value = render_tile_context(row)
        split_panel.layout.display = 'none'
        subregion_info.value = (
            f"<b style='color:#ffd700'>Labeling sub-region "
            f"{si+1}/{len(state['split_regions'])} of cluster #{idx}</b>"
        )
        subregion_nav.layout.display = ''

    dist_img.value = render_class_distribution()

# ── Wire up ────────────────────────────────────────────────────────────────────
for b in lbl_btns: b.on_click(on_label)
confirm_btn.on_click(on_confirm)
noise_btn.on_click(on_noise)
prev_btn.on_click(on_prev)
skip_btn.on_click(on_next)
next_btn.on_click(on_next)
jump_btn.on_click(on_jump)
jump_class_btn.on_click(on_jump_class)
new_btn.on_click(on_new_class)
split_btn.on_click(on_split)
add_region_btn.on_click(on_add_region)
split_preview_btn.on_click(on_split_preview)
split_apply_btn.on_click(on_split_apply)
split_cancel_btn.on_click(on_split_cancel)
cancel_split_btn.on_click(on_cancel_split_lb)

refresh()
display(main_ui)

## Labeling progress summary

In [ ]:
inv = pd.read_csv(INVENTORY_CSV)
labeled = inv[inv['final_label'].notna()]
lnames  = all_label_names()

print(f"Total clusters:  {len(inv)}")
print(f"Labeled:         {len(labeled)} ({100*len(labeled)/max(len(inv),1):.0f}%)")
print()
print(f"{'Class':<22} {'Labeled':>8} {'Target':>8} {'Progress':>10}")
print('-' * 52)
for code, target in sorted(LABEL_TARGETS.items()):
    count = int((labeled['final_label'] == code).sum())
    name  = lnames.get(code, str(code))
    pct   = f"{100*count//max(target,1)}%" if target else 'n/a'
    bar   = '█' * (count * 20 // max(target, 1)) if target else ''
    print(f"  {name:<20} {count:>8} {target:>8}   {bar} {pct}")